In [ ]:
#load file:

import pandas as pd 

flow_df_raw = pd.read_excel(r"Nutlin-3_flow.xlsx",header=0)



In [ ]:
#Calculate death in new row as combination of rhs 

flow_df_raw['Death_Percent'] = flow_df_raw['UR'] + flow_df_raw['LR']



In [ ]:
#Get N174 Death numbers, aka baseline killing in no peptide cells...
import numpy as np 
N174_df = flow_df_raw[(flow_df_raw['Peptide'] == 'N')]

N174_DMSO_death  = N174_df[N174_df['Treatment']=='D']['Death_Percent'].to_list()
N174_nut_death  = N174_df[N174_df['Treatment']=='N']['Death_Percent'].to_list()

mean_death_diff_N174 = np.mean(N174_nut_death) - np.mean(N174_DMSO_death)
#calc baseline death

from scipy import stats

ttest_N174 = stats.ttest_ind(N174_DMSO_death, N174_nut_death, alternative='less')

print(ttest_N174)




In [ ]:
#df just for peptides

flow_df_peptides = flow_df_raw[flow_df_raw['Peptide']!='N']



In [ ]:
#death for each peptide:

Statistics_DF = pd.DataFrame(columns=['Peptide', 'Death DMSO', 'Death nutlin', 'Average Killing Difference (Nutlin-DMSO)', 'Killing Relative to N174 (Killing N174 - Killing Peptide)', 'variance (nut)', 'variance (dmso)'])

for peptide in flow_df_peptides['Peptide'].unique():
    
    #DMSO death
    DMSO_death =  flow_df_peptides[(flow_df_peptides['Peptide']==peptide)&(flow_df_peptides['Treatment']=='D')]['Death_Percent'].to_list()
    #nut death
    NUT_death =flow_df_peptides[(flow_df_peptides['Peptide']==peptide)&(flow_df_peptides['Treatment']=='N')]['Death_Percent'].to_list()
    Statistics_DF.loc[len(Statistics_DF)] = [peptide,DMSO_death, NUT_death, np.mean(NUT_death) - DMSO_death,  (np.mean(NUT_death) - DMSO_death) - mean_death_diff_N174, np.var(NUT_death), np.var(DMSO_death)]
    
Statistics_DF.loc[len(Statistics_DF)] = ['N174', N174_DMSO_death, N174_nut_death, np.mean(N174_nut_death)- N174_DMSO_death, 'NA', np.var(N174_nut_death), np.var(N174_DMSO_death)]




In [ ]:
import seaborn as sns 
import matplotlib.pyplot as plt 

Exploded_Df = Statistics_DF.explode('Average Killing Difference (Nutlin-DMSO)')
g = sns.barplot(data=Exploded_Df, x='Peptide', y='Average Killing Difference (Nutlin-DMSO)', alpha=0.5)
sns.swarmplot(data=Exploded_Df, x='Peptide', y='Average Killing Difference (Nutlin-DMSO)', ax=g.axes)

plt.ylabel('%Dead by Nutlin - %Dead with DMSO')

sns.despine()



In [ ]:
Exploded_Along_Raw = Statistics_DF.explode(['Death nutlin', 'Death DMSO'], ignore_index=True)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt


# ============================================================
# 1. Prepare the data
# ============================================================

df = Exploded_Along_Raw[
    ['Peptide', 'Death DMSO', 'Death nutlin']
].copy()

# Convert death measurements to numeric
df['Death DMSO'] = pd.to_numeric(
    df['Death DMSO'],
    errors='coerce'
)

df['Death nutlin'] = pd.to_numeric(
    df['Death nutlin'],
    errors='coerce'
)

# Remove rows with missing values
df = df.dropna(
    subset=['Peptide', 'Death DMSO', 'Death nutlin']
)


# ============================================================
# 2. Convert wide → long
# ============================================================

df_long = pd.melt(
    df,
    id_vars='Peptide',
    value_vars=['Death DMSO', 'Death nutlin'],
    var_name='Treatment',
    value_name='Death'
)

# Clean treatment names
df_long['Treatment'] = df_long['Treatment'].replace({
    'Death DMSO': 'DMSO',
    'Death nutlin': 'Nutlin'
})

# Make sure Death is numeric
df_long['Death'] = pd.to_numeric(
    df_long['Death'],
    errors='coerce'
)

# Convert factors to categorical
df_long['Peptide'] = df_long['Peptide'].astype(str)
df_long['Treatment'] = df_long['Treatment'].astype(str)


print("Number of observations:", len(df_long))

print("\nReplicates per peptide:")
print(
    df_long
    .groupby(['Peptide', 'Treatment'])
    .size()
)


# ============================================================
# 3. TWO-WAY ANOVA
#
# Death ~ Treatment * Peptide
#
# Tests:
#   Treatment
#   Peptide
#   Treatment × Peptide
# ============================================================

model = ols(
    'Death ~ C(Treatment) * C(Peptide)',
    data=df_long
).fit()

anova = sm.stats.anova_lm(
    model,
    typ=2
)

print("\n======================================")
print("TWO-WAY ANOVA")
print("======================================")

print(anova)

print("\nTreatment p =",
      anova.loc['C(Treatment)', 'PR(>F)'])

print("Peptide p =",
      anova.loc['C(Peptide)', 'PR(>F)'])

print("Treatment × Peptide p =",
      anova.loc[
          'C(Treatment):C(Peptide)',
          'PR(>F)'
      ])


# ============================================================
# 4. Calculate the Nutlin effect for every replicate
#
# IMPORTANT:
# We are NOT treating DMSO/Nutlin as paired.
#
# This is simply:
#
#     Nutlin effect = Nutlin mean - DMSO mean
#
# calculated at the group level below.
# ============================================================

summary = (
    df_long
    .groupby(
        ['Peptide', 'Treatment'],
        observed=True
    )['Death']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)

print("\n======================================")
print("GROUP SUMMARY")
print("======================================")

print(summary)


# ============================================================
# 5. Calculate mean Nutlin - DMSO effect
# ============================================================

effect = (
    summary
    .pivot(
        index='Peptide',
        columns='Treatment',
        values='mean'
    )
    .reset_index()
)

effect['Nutlin_effect'] = (
    effect['Nutlin'] -
    effect['DMSO']
)

print("\n======================================")
print("NUTLIN EFFECT")
print("======================================")

print(
    effect.sort_values(
        'Nutlin_effect',
        ascending=False
    )
)


# ============================================================
# 6. Identify N174
# ============================================================

n174_effect = effect.loc[
    effect['Peptide'] == 'N174',
    'Nutlin_effect'
].iloc[0]

print("\nN174 Nutlin effect =", n174_effect)


# ============================================================
# 7. Compare EACH peptide's Nutlin effect against N174
#
# Because DMSO and Nutlin are NOT paired, we use an
# independent Welch t-test on the replicate-level
# treatment response distributions.
#
# This is a secondary analysis after the ANOVA interaction.
# ============================================================

results = []

ttest_df = pd.DataFrame(columns=['Peptide', 'DMSO (mean)', 'Nutlin (mean)', 'pval'])


for peptide in df_long['Peptide'].unique():

    # Skip N174
    if peptide == 'N174':
        continue

    peptide_data = df_long[
        df_long['Peptide'] == peptide
    ]

    n174_data = df_long[
        df_long['Peptide'] == 'N174'
    ]

    # Get independent DMSO/Nutlin groups
    peptide_dmso = peptide_data.loc[
        peptide_data['Treatment'] == 'DMSO',
        'Death'
    ]

    peptide_nutlin = peptide_data.loc[
        peptide_data['Treatment'] == 'Nutlin',
        'Death'
    ]

    n174_dmso = n174_data.loc[
        n174_data['Treatment'] == 'DMSO',
        'Death'
    ]

    n174_nutlin = n174_data.loc[
        n174_data['Treatment'] == 'Nutlin',
        'Death'
    ]

    # --------------------------------------------------------
    # Estimate treatment effect and its variance
    # without assuming DMSO/Nutlin pairing.
    #
    # Var(Nutlin-DMSO) =
    #       Var(Nutlin)/n_Nutlin
    #       +
    #       Var(DMSO)/n_DMSO
    # --------------------------------------------------------

    
    
    peptide_effect = (
        peptide_nutlin.mean() -
        peptide_dmso.mean()
    )

    n174_effect_value = (
        n174_nutlin.mean() -
        n174_dmso.mean()
    )

    ttest_df.loc[len(ttest_df)] = [peptide, peptide_dmso.mean(),peptide_nutlin.mean(), stats.ttest_ind(peptide_dmso, peptide_nutlin).pvalue]
    
    # Standard error of peptide treatment effect
    peptide_se = np.sqrt(
        peptide_nutlin.var(ddof=1) / len(peptide_nutlin)
        +
        peptide_dmso.var(ddof=1) / len(peptide_dmso)
    )

    # Standard error of N174 treatment effect
    n174_se = np.sqrt(
        n174_nutlin.var(ddof=1) / len(n174_nutlin)
        +
        n174_dmso.var(ddof=1) / len(n174_dmso)
    )

    # Difference between treatment effects
    difference = (
        peptide_effect -
        n174_effect_value
    )

    # SE of difference between treatment effects
    difference_se = np.sqrt(
        peptide_se**2 +
        n174_se**2
    )

    # Test statistic
    t_stat = difference / difference_se

    # Welch-Satterthwaite degrees of freedom
    df_welch = (
        (peptide_se**2 + n174_se**2)**2
        /
        (
            (peptide_se**4 / (len(peptide_nutlin) + len(peptide_dmso) - 2))
            +
            (n174_se**4 / (len(n174_nutlin) + len(n174_dmso) - 2))
        )
    )

    p_value = 2 * stats.t.sf(
        abs(t_stat),
        df=df_welch
    )

    results.append({
        'Peptide': peptide,
        'Peptide_Nutlin_effect': peptide_effect,
        'N174_Nutlin_effect': n174_effect_value,
        'Difference_from_N174': difference,
        't_stat': t_stat,
        'df': df_welch,
        'p_value': p_value
    })


results_df = pd.DataFrame(results)


# ============================================================
# 8. FDR correction
#
# We are testing MANY peptides against N174,
# so correct for multiple comparisons.
# ============================================================

results_df['FDR'] = multipletests(
    results_df['p_value'],
    method='fdr_bh'
)[1]

results_df['Significant'] = (
    results_df['FDR'] < 0.05
)


# Sort strongest effects first
results_df = results_df.sort_values(
    'Difference_from_N174',
    ascending=False
)


# ============================================================
# 9. Display final results
# ============================================================

print("\n======================================")
print("PEPTIDES VS N174")
print("======================================")

print(
    results_df[
        [
            'Peptide',
            'Peptide_Nutlin_effect',
            'N174_Nutlin_effect',
            'Difference_from_N174',
            'p_value',
            'FDR',
            'Significant'
        ]
    ]
)


# ============================================================
# 10. Plot
# ============================================================

plot_df = results_df.sort_values(
    'Difference_from_N174'
)

plt.figure(figsize=(10, 12))

sns.barplot(
    data=plot_df,
    x='Difference_from_N174',
    y='Peptide',
    hue='Significant',
    palette={
        True: 'crimson',
        False: 'lightgray'
    },
    legend=False
)

plt.axvline(
    0,
    color='black',
    linestyle='--'
)

plt.xlabel(
    'Nutlin effect relative to N174\n'
    '(Peptide effect − N174 effect)'
)

plt.ylabel('Peptide')

plt.title(
    'Peptide-specific Nutlin Response Relative to N174'
)

plt.tight_layout()
plt.show()

In [ ]:
print(Exploded_Along_Raw[['Peptide', 'Death DMSO', 'Death nutlin']].head(12))

In [ ]:
print(Exploded_Along_Raw.groupby('Peptide')[['Death DMSO', 'Death nutlin']].count().head(10))

In [ ]:
print(df_long)


sns.stripplot(df_long, x='Death', hue='Peptide')
plt.show()
sns.stripplot(df_long, x='Death', hue='Treatment')

In [ ]:

# 1. Map labels to DataFrame
peptide_mapping = {
    "1": r"$\text{VPg}^{85-133}$",     "T01": r"$\text{VPg}^{85-133}$",
    "2": r"$\text{UL25}^{264-312}$",   "T02": r"$\text{UL25}^{264-312}$",
    "3": r"$\text{MCP}^{432-480}$",    "T03": r"$\text{MCP}^{432-480}$",
    "4": r"$\text{MCP}^{552-600}$",    "T04": r"$\text{MCP}^{552-600}$",
    "5": r"$\text{UL140}^{120-168}$",  "T05": r"$\text{UL140}^{120-168}$",
    "6": r"$\text{pTP}^{312-360}$",    "T06": r"$\text{pTP}^{312-360}$",
    "7": r"$\text{OCP}^{384-432}$",    "T07": r"$\text{OCP}^{384-432}$",
    "10": r"$\text{LANA}^{96-144}$",   "T10": r"$\text{LANA}^{96-144}$","A2": r"$\text{VP11/12}^{96-144}$", # <-- Same value
    "B6": r"$\text{ORF25}^{1080-1128}$",
    "N174": "N174"
}

df_long['Peptide_Mapped'] = df_long['Peptide'].astype(str).map(peptide_mapping).fillna(df_long['Peptide'])

explicit_order = [
    peptide_mapping["1"], peptide_mapping["2"], peptide_mapping["3"], 
    peptide_mapping["4"], peptide_mapping["5"], peptide_mapping["7"], 
    peptide_mapping["6"], peptide_mapping["10"], peptide_mapping["A2"], 
    peptide_mapping["B6"], peptide_mapping["N174"]
]

print(df_long.to_string())

fig, ax = plt.subplots(figsize=(10, 6))

sns.stripplot(
    data=df_long, 
    x='Peptide_Mapped', 
    y='Death', 
    order=explicit_order,
    dodge=True, 
    hue="Treatment", 
    jitter=False, 
    size=8, 
    edgecolor='k',
    linewidth=1.5,
    alpha=0.7,
    ax=ax
)

plt.ylabel("% Dead")
plt.xlabel("Peptide or control", labelpad=35) 

# Since 'order' sets the labels, we just adjust the rotation and alignment properties.
ax.tick_params(axis='x', rotation=45, labelsize=8.5)
for label in ax.get_xticklabels():
    label.set_horizontalalignment('right') # 'right' alignment looks best for 45-degree angles


'''
sns.despine(
    top=True, right=True,
    offset={'left': 13, 'bottom': 20}, 
    trim=False
)
'''
sns.despine(top=True,right=True,trim=False)

ttest_df['qval'] = stats.false_discovery_control(ttest_df['pval'], method='bh')

# Your original horizontal q-value layout remaining completely untouched
x = -0.27
y = -0.2
#for qval in ttest_df['qval']:
#    plt.text(x=x, y=y, s=f"{qval:.2f}", size=10, color='grey')
#    x += 1.01

#plt.text(x=10.5, y=-0.2, s="q-Value", size=10)
plt.tight_layout()


plt.savefig(r"Nutlin_Flow.pdf", dpi=300)



In [ ]:
#plot results in a way similar to the original RITA treatment + Flow experiment:

print(results_df.head())

cmapping = {"N174": "Gray"}

sns.barplot(results_df, x='Peptide', y='Difference_from_N174')
plt.ylabel("% Cells Dead with Nutlin in Peptide Relative to Control")
plt.xlabel("Peptide-Containing Cell Line")